In [1]:
print("hello QUBIT CREW")

hello QUBIT CREW


In [3]:
%pip install -q qiskit qiskit-optimization

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 24.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.1/237.1 kB 14.2 MB/s eta 0:00:00


In [4]:
import qiskit
print("qiskit version ", qiskit.__version__)
print("installation successful")

qiskit version  2.5.2
installation successful


In [5]:
from itertools import product

# Teaching example — not official hackathon data
items = ["Laptop", "Camera", "Headphones"]
weights = [3, 4, 2]
values = [4, 5, 3]
capacity = 7

best_value = 0
best_choice = None

# Try every possible combination of 0s and 1s
for choice in product([0, 1], repeat=len(items)):
    total_weight = sum(
        choice[i] * weights[i] for i in range(len(items))
    )
    total_value = sum(
        choice[i] * values[i] for i in range(len(items))
    )

    if total_weight <= capacity and total_value > best_value:
        best_value = total_value
        best_choice = choice

print("Best selection:")
for i in range(len(items)):
    if best_choice[i] == 1:
        print("-", items[i])

print("Total weight:", sum(
    best_choice[i] * weights[i] for i in range(len(items))
), "kg")
print("Total value:", best_value)

Best selection:
- Laptop
- Camera
Total weight: 7 kg
Total value: 9


In [6]:
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo

# Create the optimization problem
problem = QuadraticProgram("Knapsack_Example")

# Binary decision variables: 0 = don't select, 1 = select
problem.binary_var("laptop")
problem.binary_var("camera")
problem.binary_var("headphones")

# Maximize total value
problem.maximize(
    linear={
        "laptop": 4,
        "camera": 5,
        "headphones": 3
    }
)

# Total weight must be at most 7 kg
problem.linear_constraint(
    linear={
        "laptop": 3,
        "camera": 4,
        "headphones": 2
    },
    sense="<=",
    rhs=7,
    name="weight_limit"
)

print("Original constrained problem:")
print(problem.prettyprint())

# Convert the constrained problem into a QUBO
converter = QuadraticProgramToQubo()
qubo = converter.convert(problem)

print("\nConverted QUBO problem:")
print(qubo.prettyprint())

print("\nQUBO conversion successful!")

Original constrained problem:
Problem name: Knapsack_Example

Maximize
  5*camera + 3*headphones + 4*laptop

Subject to
  Linear constraints (1)
    4*camera + 2*headphones + 3*laptop <= 7  'weight_limit'

  Binary variables (3)
    laptop camera headphones


Converted QUBO problem:
Problem name: Knapsack_Example

Minimize
  208*camera^2 + 208*camera*headphones + 104*camera*weight_limit@int_slack@0
  + 208*camera*weight_limit@int_slack@1 + 416*camera*weight_limit@int_slack@2
  + 52*headphones^2 + 52*headphones*weight_limit@int_slack@0
  + 104*headphones*weight_limit@int_slack@1
  + 208*headphones*weight_limit@int_slack@2 + 312*laptop*camera
  + 156*laptop*headphones + 117*laptop^2 + 78*laptop*weight_limit@int_slack@0
  + 156*laptop*weight_limit@int_slack@1 + 312*laptop*weight_limit@int_slack@2
  + 13*weight_limit@int_slack@0^2
  + 52*weight_limit@int_slack@0*weight_limit@int_slack@1
  + 104*weight_limit@int_slack@0*weight_limit@int_slack@2
  + 52*weight_limit@int_slack@1^2
  + 208*weig

In [7]:
from qiskit_optimization import QuadraticProgram
from qiskit.circuit.library import QAOAAnsatz
from qiskit import transpile

# Reuse the QUBO problem from Step 7
# The variable `qubo` should already exist.

# Convert QUBO to an Ising Hamiltonian
ising_operator, offset = qubo.to_ising()

print("Ising Hamiltonian:")
print(ising_operator)
print("\nEnergy offset:", offset)
print("Number of qubits:", ising_operator.num_qubits)

# Build a one-layer QAOA circuit
qaoa_circuit = QAOAAnsatz(
    cost_operator=ising_operator,
    reps=1
)

print("\nQAOA circuit:")
print(qaoa_circuit.draw(output="text"))

print("\nCircuit qubits:", qaoa_circuit.num_qubits)
print("Circuit depth:", qaoa_circuit.depth())
print("Circuit operations:", qaoa_circuit.count_ops())

Ising Hamiltonian:
SparsePauliOp(['IIIIIZ', 'IIIIZI', 'IIIZII', 'IIZIII', 'IZIIII', 'ZIIIII', 'IIIIZZ', 'IIIZIZ', 'IIZIIZ', 'IZIIIZ', 'ZIIIIZ', 'IIIZZI', 'IIZIZI', 'IZIIZI', 'ZIIIZI', 'IIZZII', 'IZIZII', 'ZIIZII', 'IZZIII', 'ZIZIII', 'ZZIIII'],
              coeffs=[-37. +0.j, -49.5+0.j, -24.5+0.j, -13. +0.j, -26. +0.j, -52. +0.j,
  78. +0.j,  39. +0.j,  19.5+0.j,  39. +0.j,  78. +0.j,  52. +0.j,
  26. +0.j,  52. +0.j, 104. +0.j,  13. +0.j,  26. +0.j,  52. +0.j,
  13. +0.j,  26. +0.j,  52. +0.j])

Energy offset: 169.5
Number of qubits: 6

QAOA circuit:
     ┌──────────────────┐
q_0: ┤0                 ├
     │                  │
q_1: ┤1                 ├
     │                  │
q_2: ┤2                 ├
     │  QAOA(γ[0],β[0]) │
q_3: ┤3                 ├
     │                  │
q_4: ┤4                 ├
     │                  │
q_5: ┤5                 ├
     └──────────────────┘

Circuit qubits: 6
Circuit depth: 1
Circuit operations: OrderedDict({'QAOA': 1})


In [8]:
%pip install -q qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 102.2 MB/s eta 0:00:00


In [9]:
from qiskit import transpile
from qiskit.primitives import StatevectorSampler

# Assign trial values to the QAOA parameters.
# These are example parameters, not optimized values.
parameters = list(qaoa_circuit.parameters)
print("Number of QAOA parameters:", len(parameters))

values = [0.5, 0.7]  # one gamma and one beta for reps=1
if len(parameters) != len(values):
    raise ValueError(
        "Unexpected parameter count. Tell me the printed count."
    )

bound_circuit = qaoa_circuit.assign_parameters(
    dict(zip(parameters, values))
)

# Add measurements so the simulator can record bit strings.
measured_circuit = bound_circuit.copy()
measured_circuit.measure_all()

# Run locally on a simulator for 1,000 shots.
sampler = StatevectorSampler(seed=42)
job = sampler.run([measured_circuit], shots=1000)
result = job.result()

counts = result[0].data.meas.get_counts()
print("Measured bit strings:")
print(counts)

Number of QAOA parameters: 2


/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.13/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


Measured bit strings:
{'110001': 90, '011111': 67, '110101': 27, '101110': 12, '000110': 27, '111101': 14, '001010': 54, '011001': 8, '111000': 59, '101100': 36, '110011': 1, '001111': 12, '100100': 48, '000101': 32, '110100': 25, '101011': 15, '011000': 19, '111100': 4, '001100': 10, '101101': 18, '111011': 2, '010110': 15, '001011': 21, '010100': 9, '011010': 12, '010010': 44, '000001': 14, '001000': 17, '010011': 5, '000011': 11, '001110': 10, '011011': 24, '110010': 8, '101111': 14, '100010': 7, '111010': 23, '001101': 5, '100101': 1, '100111': 32, '011101': 3, '100001': 12, '000111': 15, '100000': 1, '100011': 18, '000010': 1, '110000': 6, '000100': 5, '010101': 8, '111111': 12, '010001': 19, '110110': 10, '001001': 5, '101000': 2, '010000': 4, '011110': 4, '101001': 9, '000000': 7, '110111': 1, '101010': 3, '010111': 2, '111110': 1}


In [11]:
# Decode each measured bit string back to the original
# knapsack variables and evaluate its real weight and value.

candidate_results = []

for bitstring, frequency in counts.items():
    # Qiskit displays bit strings in reverse qubit order.
    # Convert the string to the QUBO variable vector order.
    x_qubo = [int(bit) for bit in bitstring[::-1]]

    # Convert the QUBO assignment back to the original problem.
    original_x = converter.interpret(x_qubo)

    # First three variables are laptop, camera, headphones.
    selection = [int(round(float(v))) for v in original_x[:3]]

    weight = sum(
        selection[i] * weights[i] for i in range(3)
    )
    value = sum(
      selection[i] * values[i] for i in range(3)
    )

    valid = weight <= capacity

    candidate_results.append({
        "bitstring": bitstring,
        "shots": frequency,
        "selection": selection,
        "weight": weight,
        "value": value,
        "valid": valid
    })

# Display the most frequently measured candidates first.
candidate_results.sort(key=lambda row: row["shots"], reverse=True)

for row in candidate_results[:10]:
    print(
        f"Bits={row['bitstring']} | "
        f"Selection={row['selection']} | "
        f"Weight={row['weight']} | "
        f"Value={row['value']} | "
        f"Valid={row['valid']} | "
        f"Shots={row['shots']}"
    )

valid_candidates = [
    row for row in candidate_results if row["valid"]
]

if valid_candidates:
    best = max(valid_candidates, key=lambda row: row["value"])
    print("\nBest valid sampled candidate:", best["selection"])
    print("Weight:", best["weight"], "kg")
    print("Value:", best["value"])
else:
    print("\nNo valid candidate was sampled.")

IndexError: list index out of range

In [12]:
# Qubit Crew — PS-03 Knapsack
# Emergency clean working demo
# Teaching data only, not official hackathon data

from itertools import product

items = ["Laptop", "Camera", "Headphones"]
weights = [3, 4, 2]
values = [4, 5, 3]
capacity = 7

valid = []

for x in product([0, 1], repeat=3):
    weight = sum(x[i] * weights[i] for i in range(3))
    value = sum(x[i] * values[i] for i in range(3))

    if weight <= capacity:
        valid.append((value, weight, x))

valid.sort(reverse=True)

print("QUBIT CREW | KNAPSACK DEMO")
print("Capacity:", capacity, "kg")
print("Best valid solution:")

value, weight, x = valid[0]

for i in range(3):
    print(items[i], ":", "SELECTED" if x[i] else "NOT SELECTED")

print("Total weight:", weight, "kg")
print("Total value:", value)
print("All constraints satisfied:", weight <= capacity)

QUBIT CREW | KNAPSACK DEMO
Capacity: 7 kg
Best valid solution:
Laptop : SELECTED
Camera : SELECTED
Headphones : NOT SELECTED
Total weight: 7 kg
Total value: 9
All constraints satisfied: True
